# 🎙️ Atomic Step 6b: Speaker-Wise Transcription (Local Gemma-3n-e4b)

This notebook runs Gurmukhi script Punjabi transcription locally using the **Gemma 3n E4B** multimodal language model (`google/gemma-3n-e4b-it`) on GPU, exporting formatted conversation transcripts (TXT, JSON, MD) and speaker-segregated texts to Google Drive.

In [ ]:
# 1. Uninstall incompatible torchao to prevent conflicts
!pip uninstall -y -q torchao

# 2. Install Transformers, Accelerate, and audio utilities
!pip install -q transformers accelerate librosa soundfile --prefer-binary
print("[SUCCESS] Dependencies installed!")

In [ ]:
import torch

# Set device and dtype
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.bfloat16 if "cuda" in device else torch.float32
print(f"[INFO] Using device: {device} | Dtype: {torch_dtype}")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.")
except Exception:
    print("[INFO] Already mounted or skipped.")

## ⚙️ Parameters

In [ ]:
# @markdown ### 📁 Audio & Timeline Inputs
cleaned_audio_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Cleaned_Audio/MarauliKhurad1_cleaned.wav" # @param {type:"string"}
refined_timeline_json_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Diarization_Outputs/MarauliKhurad1/MarauliKhurad1_timeline.json" # @param {type:"string"}
transcription_output_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Diarization_Transcripts/" # @param {type:"string"}

# @markdown ### 🤖 Model Configs
gemma_model_id = "google/gemma-3n-e4b-it" # @param {type:"string"}

import os
if not os.path.exists(cleaned_audio_path):
    print(f"[ERROR] Cleaned audio not found at: '{cleaned_audio_path}'")
elif not os.path.exists(refined_timeline_json_path):
    print(f"[ERROR] Refined timeline JSON not found at: '{refined_timeline_json_path}'")
else:
    audio_filename = os.path.basename(cleaned_audio_path)
    audio_name_only = audio_filename.replace("_cleaned.wav", "").replace(".wav", "")
    os.makedirs(transcription_output_folder, exist_ok=True)
    print(f"[SUCCESS] Validated paths. Output transcripts will be saved in: {transcription_output_folder}")

## 🎙️ Step 1: Run Local Gemma-3n Transcription

In [ ]:
import soundfile as sf
import librosa
import json
import os
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

if os.path.exists(cleaned_audio_path) and os.path.exists(refined_timeline_json_path):
    # Create subfolder for output transcripts
    specific_transcript_folder = os.path.join(transcription_output_folder, audio_name_only)
    os.makedirs(specific_transcript_folder, exist_ok=True)
    
    # Load timeline JSON
    with open(refined_timeline_json_path, "r", encoding="utf-8") as f:
        speaker_segments = json.load(f)
        
    # Load cleaned audio
    print(f"Loading cleaned audio file for transcription: '{cleaned_audio_path}'")
    y, sr = librosa.load(cleaned_audio_path, sr=16000, mono=True)
    
    print(f"\n--- Transcribing Segments via Local Gemma 3n Model ({gemma_model_id}) ---")
    diarized_transcript_entries = []
    speaker_texts = {}
    
    # Time formatting helper
    def format_time(seconds):
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int((seconds - int(seconds)) * 10)
        if hours > 0:
            return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:01d}"
        else:
            return f"{minutes:02d}:{secs:02d}.{millis:01d}"
            
    # Load HF Token
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except (ImportError, ValueError):
        HF_TOKEN = os.environ.get("HF_TOKEN")
        
    if not HF_TOKEN:
        print("[WARNING] HF_TOKEN not found in environment. Model download might fail if restricted.")
        
    print(f"Loading processor and model for {gemma_model_id}...")
    processor = AutoProcessor.from_pretrained(gemma_model_id, token=HF_TOKEN)
    model = AutoModelForMultimodalLM.from_pretrained(
        gemma_model_id,
        token=HF_TOKEN,
        torch_dtype=torch_dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model.eval()
    print("[SUCCESS] Gemma-3n model loaded successfully!")
    
    ASR_PROMPT = (
        "Transcribe the following speech segment in Punjabi into "
        "Gurmukhi text. Output only the raw transcript, with no "
        "introductory text or newlines."
    )
    
    # Run transcription loop
    for idx, entry in enumerate(speaker_segments):
        start_sec = entry['start']
        end_sec = entry['end']
        speaker = entry['speaker']
        
        duration = end_sec - start_sec
        if duration < 0.3:
            continue
            
        # Slice chunk in-memory
        start_sample = int(start_sec * sr)
        end_sample = int(end_sec * sr)
        chunk = y[start_sample:end_sample]
        
        # Prepare message for Gemma 3n
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": chunk},
                    {"type": "text", "text": ASR_PROMPT},
                ],
            },
        ]
        
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[-1]
        
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                repetition_penalty=1.1
            )
            
        response_ids = generated_ids[0][input_len:]
        text = processor.decode(response_ids, skip_special_tokens=True).strip()
        
        if text:
            time_str = f"[{format_time(start_sec)} - {format_time(end_sec)}]"
            print(f"{time_str} {speaker}: {text}")
            
            diarized_transcript_entries.append({
                "time": time_str,
                "speaker": speaker,
                "text": text
            })
            
            if speaker not in speaker_texts:
                speaker_texts[speaker] = []
            speaker_texts[speaker].append(text)
            
    # Export outputs
    if diarized_transcript_entries:
        txt_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_diarized_transcript.txt")
        md_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_diarized_transcript.md")
        json_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_diarized_transcript.json")
        
        # Write Text File
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(f"Diarized Transcript for: {audio_filename}\n")
            f.write("=" * 60 + "\n\n")
            for entry in diarized_transcript_entries:
                f.write(f"{entry['time']} {entry['speaker']}: {entry['text']}\n")
                
        # Write Markdown File
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(f"# 🎙️ Diarized Transcript: {audio_filename}\n\n")
            f.write(f"Generated using local Gemma-3n model ({gemma_model_id}).\n\n")
            for entry in diarized_transcript_entries:
                f.write(f"> **{entry['speaker']}** `{entry['time']}`  \n")
                f.write(f"> {entry['text']}\n\n")
                
        # Write JSON File
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(diarized_transcript_entries, f, indent=4, ensure_ascii=False)
            
        # Write Speaker Segregated Files
        for speaker, texts in speaker_texts.items():
            spk_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_{speaker}_transcript.txt")
            with open(spk_path, "w", encoding="utf-8") as f:
                f.write("\n".join(texts))
                
        print(f"\n[SUCCESS] Transcription files exported successfully to: '{specific_transcript_folder}'")
    else:
        print("[WARNING] No speech content could be transcribed.")